In [ ]:
%pip install kaggle pillow scikit-learn matplotlib seaborn tqdm

hello


# Installation and Setup
We will be importing these libraries for the said purpose:
- Json -
- Shutil -

In [ ]:
import os
import json
import shutil
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

From there, we will mount to our google drive to get our data that is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After this, we will set our paths accordingly to our google drive into these variables

In [ ]:
dataset_path = Path('/content/drive/MyDrive/Machine_Learning/Assignment/dataset')
training_data_path = dataset_path / 'training_data'
validation_data_path = dataset_path / 'validation_data'
testing_data_path = dataset_path / 'test_data'

# Data Preparation & Processing Pipeline
After installing and setting up the folder paths, we will need to inspect and understand our dataset. From there, we will need to process our data to see if there's any issues with it such as defected or corrupted images.

## Count number of images
In our given dataset, it is split across 3 different folders. We have our training set, validation set and our test set. Our training set and validation set contains several folders. These folders are our classes, representing our type of plants.

In [ ]:
# Count the number of images in each data folder by going through each class sub-folder
def count_images_per_class(data_directory):
    class_counts = {}
    for class_folder in sorted(os.listdir(data_directory)):
        class_path = os.path.join(data_directory, class_folder)
        if os.path.isdir(class_path):
            num_images = len([f for f in os.listdir(class_path)
                            if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            class_counts[class_folder] = num_images
    return class_counts

In [ ]:
# Count images in each split
# For the test set, we will use the count from check_corrupted_images if it doesn't have class subfolders.
# The test_counts will be 0 if there are no subdirectories.
train_counts = count_images_per_class(training_data_path)
valid_counts = count_images_per_class(validation_data_path)
test_counts = count_images_per_class(testing_data_path)

print(f"Training set: {sum(train_counts.values())} images across {len(train_counts)} classes")
print(f"Validation set: {sum(valid_counts.values())} images across {len(valid_counts)} classes")

# Check if test_counts is empty, if so, use the total valid images from the earlier check
if not test_counts and 'test_valid' in globals():
    print(f"Test set: {len(test_valid)} images (total, no class subfolders found)")
else:
    print(f"Test set: {sum(test_counts.values())} images across {len(test_counts)} classes")


## Check for corrupted images
It's possible for our dataset to include corrupt or defected images. We need to verify if this is or isn't the case.

In [ ]:
def check_corrupted_images(folder_path):
    """
    Scan all images in a folder and check if they can be opened.
    Returns lists of valid and corrupted image paths.
    """
    valid_images = []
    corrupted_images = []

    # Walk through all subfolders (class folders)
    for root, directories, files in os.walk(folder_path):
        for filename in files:
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, filename)
                try:
                    # Try to open and verify the image
                    with Image.open(img_path) as img:
                        img.verify()  # This checks if file is corrupted
                    valid_images.append(img_path)
                except Exception as e:
                    corrupted_images.append((img_path, str(e)))

    return valid_images, corrupted_images

In [ ]:
# Run on each split
print("Checking TRAIN folder...")
train_valid, train_corrupted = check_corrupted_images(TRAIN_PATH)
print(f"  Valid: {len(train_valid)}, Corrupted: {len(train_corrupted)}")

print("Checking VALID folder...")
valid_valid, valid_corrupted = check_corrupted_images(VALID_PATH)
print(f"  Valid: {len(valid_valid)}, Corrupted: {len(valid_corrupted)}")

print("Checking TEST folder...")
test_valid, test_corrupted = check_corrupted_images(TEST_PATH)
print(f"  Valid: {len(test_valid)}, Corrupted: {len(test_corrupted)}")

# Show corrupted files if any
if train_corrupted:
    print("\n⚠️ Corrupted files in TRAIN:")
    for path, error in train_corrupted:
        print(f"  {path}: {error}")

## Get image statistics

In [ ]:
def get_image_stats(folder_path):
    """
    Collect dimension and file size statistics for all images.
    """
    dimensions = []
    file_sizes = []

    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, filename)
                try:
                    with Image.open(img_path) as img:
                        dimensions.append(img.size)  # (width, height)
                    file_sizes.append(os.path.getsize(img_path))
                except:
                    pass

    return dimensions, file_sizes

In [ ]:
# Get stats for training set
dimensions, file_sizes = get_image_stats(TRAIN_PATH)

widths = [d[0] for d in dimensions]
heights = [d[1] for d in dimensions]

print("=" * 50)
print("IMAGE DIMENSION STATISTICS (Training Set)")
print("=" * 50)
print(f"Total images: {len(dimensions)}")
print(f"\nWidth:")
print(f"  Min: {min(widths)} px")
print(f"  Max: {max(widths)} px")
print(f"  Mean: {np.mean(widths):.1f} px")
print(f"\nHeight:")
print(f"  Min: {min(heights)} px")
print(f"  Max: {max(heights)} px")
print(f"  Mean: {np.mean(heights):.1f} px")
print(f"\nFile size:")
print(f"  Min: {min(file_sizes)/1024:.1f} KB")
print(f"  Max: {max(file_sizes)/1024:.1f} KB")
print(f"  Mean: {np.mean(file_sizes)/1024:.1f} KB")

## Standardise Dimensions
Scikit-learn (and most ML models) require all inputs to have the same shape. You can't feed a 500×500 image and a 1024×1168 image into the same model.

In [ ]:
def get_class_distribution(folder_path):
    """
    Count how many images are in each class folder.
    """
    class_counts = {}

    # Each subfolder is a class (named 1, 2, 3, ... 102)
    for class_folder in sorted(Path(folder_path).iterdir()):
        if class_folder.is_dir():
            class_name = class_folder.name
            # Count images in this class
            num_images = len(list(class_folder.glob("*.jpg"))) + \
                        len(list(class_folder.glob("*.png")))
            class_counts[class_name] = num_images

    return class_counts

In [ ]:
# Get distribution
train_distribution = get_class_distribution(TRAIN_PATH)

# Convert to sorted list for analysis
class_names = sorted(train_distribution.keys(), key=lambda x: int(x))
counts = [train_distribution[c] for c in class_names]

print("=" * 50)
print("CLASS DISTRIBUTION (Training Set)")
print("=" * 50)
print(f"Number of classes: {len(train_distribution)}")
print(f"Total images: {sum(counts)}")
print(f"\nImages per class:")
print(f"  Min: {min(counts)} (Class {class_names[counts.index(min(counts))]})")
print(f"  Max: {max(counts)} (Class {class_names[counts.index(max(counts))]})")
print(f"  Mean: {np.mean(counts):.1f}")
print(f"  Std Dev: {np.std(counts):.1f}")

# Find underrepresented classes (less than mean - 1 std)
threshold = np.mean(counts) - np.std(counts)
underrepresented = [(c, train_distribution[c]) for c in class_names
                    if train_distribution[c] < threshold]

if underrepresented:
    print(f"\n⚠️ Underrepresented classes (< {threshold:.0f} images):")
    for cls, count in sorted(underrepresented, key=lambda x: x[1])[:10]:
        print(f"  Class {cls}: {count} images")

## Creating a bar chart to show the class distribution

In [ ]:
# Create bar chart of class distribution
plt.figure(figsize=(16, 6))

# Sort by class number
sorted_classes = sorted(train_distribution.keys(), key=lambda x: int(x))
sorted_counts = [train_distribution[c] for c in sorted_classes]

plt.bar(range(len(sorted_classes)), sorted_counts, color='steelblue', alpha=0.7)

# Add mean line
mean_val = np.mean(sorted_counts)
plt.axhline(y=mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.1f}')

plt.xlabel('Class Number', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.title('Training Set: Images per Class', fontsize=14, fontweight='bold')
plt.legend()

# Show every 10th class on x-axis
plt.xticks(range(0, len(sorted_classes), 10),
           [sorted_classes[i] for i in range(0, len(sorted_classes), 10)])

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

## Standardise image colour coding and normalise pixel values

In [ ]:
def preprocess_image(img_path, target_size=(128, 128)):
    """
    Load and preprocess a single image for ML.

    Steps:
    1. Load image
    2. Convert to RGB (handles grayscale/RGBA)
    3. Resize to consistent size
    4. Normalize pixel values to 0-1

    Returns: numpy array of shape (height, width, 3)
    """
    img = Image.open(img_path)

    # Convert to RGB if needed (some images might be grayscale or RGBA)
    if img.mode != 'RGB':
        img = img.convert('RGB')

    # Resize to target size
    img = img.resize(target_size, Image.LANCZOS)

    # Convert to numpy array and normalize
    img_array = np.array(img, dtype=np.float32) / 255.0

    return img_array


In [ ]:
IMG_SIZE = (64, 64)

def load_dataset(folder_path):
    X, y = [], []

    # Sort numerically, not alphabetically
    class_names = sorted(os.listdir(folder_path), key=lambda x: int(x))

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(folder_path, class_name)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            img_path = os.path.join(class_path, img_name)
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(IMG_SIZE)
                img_array = np.array(img) / 255.0
                X.append(img_array.flatten())
                y.append(label)
            except Exception as e:
                print(f"Skipped {img_path}: {e}")

    return np.array(X), np.array(y), class_names

In [ ]:
# Load all datasets
print("Loading training data...")
X_train, y_train, class_names = load_dataset(TRAIN_PATH)
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")

print("Loading validation data...")
X_valid, y_valid, _ = load_dataset(VALID_PATH)
print(f"  X_valid: {X_valid.shape}, y_valid: {y_valid.shape}")

print("Loading test data...")
# The load_dataset function expects class subfolders, but TEST_PATH has direct images.
# We will manually load X_test and create a placeholder y_test.
X_test = []
# 'test_valid' was populated by check_corrupted_images and contains paths to all valid test images.
for img_path in test_valid:
    try:
        img = Image.open(img_path).convert("RGB")
        img = img.resize(IMG_SIZE)
        img_array = np.array(img) / 255.0
        X_test.append(img_array.flatten())
    except Exception as e:
        print(f"Skipped {img_path}: {e}")
X_test = np.array(X_test)
# Create a placeholder for y_test. Actual labels would typically be loaded from a separate file.
y_test = np.zeros(len(X_test), dtype=int) # Placeholder labels (e.g., all zeros)
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

print(f"\nNumber of classes: {len(class_names)}")

# Load the processed dataset

In [ ]:
# After loading the dataset
print("=" * 50)
print("DATA LOADING SANITY CHECK")
print("=" * 50)

print(f"\nShapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  X_valid: {X_valid.shape}")
print(f"  y_valid: {y_valid.shape}")

print(f"\nLabel range:")
print(f"  Min label: {y_train.min()}")
print(f"  Max label: {y_train.max()}")
print(f"  Unique labels: {len(np.unique(y_train))}")

print(f"\nPixel value range (should be 0-1):")
print(f"  Min: {X_train.min():.4f}")
print(f"  Max: {X_train.max():.4f}")

print(f"\nClass distribution (first 10):")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in list(zip(unique, counts))[:10]:
    print(f"  Class {label}: {count} images")